In [1]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "src"), str(ROOT / "setup")]

In [2]:
from functools import partial
from dotenv import load_dotenv
from IPython.display import display

from infra.database.duckdb_client import DuckDBClient
from infra.etl import ecmwf_etl as etl
from infra.pipeline.storage.s3_client import S3Client, list_available_data, s3_mapper

In [3]:
load_dotenv(ROOT / "infra" / ".env.garage")
load_dotenv(ROOT / ".env")

True

In [ ]:
RAW_BUCKET = "commodities-raw"
REGION = "europe"
DATA = "ecmwf_hourly"

list_available_data()

ecmwf_seasonal
ecmwf_hourly


In [5]:
key = s3_mapper(DATA)
s3 = S3Client()
print(key)

ecmwf/source=opendata/model=aifs-ens/resolution=0p25/stream=enfo/type=pf/param=2t


In [6]:
# Inspect the first raw GRIB object under the prefix.
raw_sample = next(etl.extract_prefix(s3, RAW_BUCKET, key), None)
if raw_sample is None:
    print(f"No GRIB objects under {key}/")
else:
    print(raw_sample[0])
    display(etl.decode_grib(raw_sample[1], max_messages=1).head())

ecmwf/source=opendata/model=aifs-ens/resolution=0p25/stream=enfo/type=pf/param=2t/date=20261125/cycle=00/steps=024-048-072/forecast.grib2


latitude,longitude,value,message,shortName,units,typeOfLevel,level,dataDate,dataTime,stepRange,validityDate,validityTime,perturbationNumber
f64,f64,f64,i32,str,str,str,i32,i32,i32,str,i32,i32,i32
90.0,-180.0,248.65416,0,"""2t""","""K""","""heightAboveGround""",2,20261125,0,"""24""",20261126,0,35
90.0,-179.75,248.65416,0,"""2t""","""K""","""heightAboveGround""",2,20261125,0,"""24""",20261126,0,35
90.0,-179.5,248.65416,0,"""2t""","""K""","""heightAboveGround""",2,20261125,0,"""24""",20261126,0,35
90.0,-179.25,248.65416,0,"""2t""","""K""","""heightAboveGround""",2,20261125,0,"""24""",20261126,0,35
90.0,-179.0,248.65416,0,"""2t""","""K""","""heightAboveGround""",2,20261125,0,"""24""",20261126,0,35


In [7]:
# Read objects on demand to keep memory bounded.
extract_objects = partial(etl.extract_prefix, s3, RAW_BUCKET, key)
sample = next(extract_objects(), None)

In [8]:
transform_objects = partial(etl.transform_objects, bucket=RAW_BUCKET, region=REGION)
if sample is not None:
    display(next(transform_objects([sample])).frame.head())

dataDate,dataTime,validityDate,validityTime,stepRange,message,latitude,longitude,value,units,perturbationNumber,region,source_uri,db_path
i32,i32,i32,i32,str,i32,f64,f64,f64,str,i32,str,str,str
20261125,0,20261126,0,"""24""",0,50.0,-125.0,285.15416,"""K""",35,"""us48""","""s3://commodities-raw/ecmwf/sou…","""s3://commodities-raw/ecmwf/sou…"
20261125,0,20261126,0,"""24""",0,50.0,-124.75,285.15416,"""K""",35,"""us48""","""s3://commodities-raw/ecmwf/sou…","""s3://commodities-raw/ecmwf/sou…"
20261125,0,20261126,0,"""24""",0,50.0,-124.5,282.90416,"""K""",35,"""us48""","""s3://commodities-raw/ecmwf/sou…","""s3://commodities-raw/ecmwf/sou…"
20261125,0,20261126,0,"""24""",0,50.0,-124.25,281.15416,"""K""",35,"""us48""","""s3://commodities-raw/ecmwf/sou…","""s3://commodities-raw/ecmwf/sou…"
20261125,0,20261126,0,"""24""",0,50.0,-124.0,278.90416,"""K""",35,"""us48""","""s3://commodities-raw/ecmwf/sou…","""s3://commodities-raw/ecmwf/sou…"


In [9]:
db = DuckDBClient()

In [10]:
db = DuckDBClient()
loaded = etl.load_objects(transform_objects(extract_objects()), data=DATA, db=db)
print(f"Loaded {loaded} new or changed objects into {DATA} in {db.db_path}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded 2 new or changed objects into ecmwf_hourly in /Users/prasunbanerjee/Desktop/research-data/duckdb/research-data.duckdb


In [11]:
tables_to_drop = []
db.drop_tables(tables_to_drop)

Tables in /Users/prasunbanerjee/Desktop/research-data/duckdb/research-data.duckdb: ['ecmwf_hourly', 'ecmwf_hourly_loaded_objects', 'ecmwf_loaded_objects', 'ecmwf_seasonal', 'ecmwf_seasonal_loaded_objects']
